# 02 — OHLCV Analysis

Loads daily OHLCV Parquet bars from the `market-analysis` MinIO bucket and computes technical indicators.

**Requires:** At least a few days of OHLCV data produced by `make run-ohlcv-daily-ingest`.

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ta.trend import SMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

from model.minio_store import MinioStore

load_dotenv()
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
store = MinioStore(os.getenv('MINIO_ANALYSIS_BUCKET', 'market-analysis'))

# List available OHLCV partitions
partitions = sorted({
    obj.object_name.split('/asset_class=')[1].split('/')[0] + ' / ' +
    '/'.join(obj.object_name.split('/')[-4:-1])
    for obj in store.list_objects(prefix='ohlcv.bar/')
    if obj.object_name.endswith('.parquet')
})
print('Available partitions (asset_class / year / month / day):')
for p in partitions:
    print(' ', p)

In [ ]:
# Load all available OHLCV bars for a given asset class
ASSET = 'stock'   # 'stock' or 'crypto'

frames = []
for obj in store.list_objects(prefix=f'ohlcv.bar/asset_class={ASSET}/'):
    if obj.object_name.endswith('.parquet'):
        rows = store.read_parquet(obj.object_name)
        frames.append(pd.DataFrame(rows))

if not frames:
    raise RuntimeError(f'No OHLCV bars found for asset_class={ASSET}. Run make run-ohlcv-daily-ingest first.')

df = pd.concat(frames, ignore_index=True)
df['date'] = pd.to_datetime(df['time'].str[:10])
df = df.sort_values('date').reset_index(drop=True)

print(f'{len(df)} bars | {df.date.min().date()} → {df.date.max().date()} | symbols: {sorted(df.symbol.unique())}')
df.head()

In [ ]:
# Candlestick-style OHLC bar chart for one symbol
SYMBOL = df['symbol'].iloc[0]  # change to any symbol in the list above

s = df[df.symbol == SYMBOL].copy()
s = s.set_index('date').sort_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

for idx, row in s.iterrows():
    color = 'green' if row.close >= row.open else 'red'
    ax1.plot([idx, idx], [row.low, row.high], color=color, linewidth=1)
    ax1.plot([idx, idx], [row.open, row.close], color=color, linewidth=4)

ax1.set_title(f'{SYMBOL} — daily OHLC')
ax1.set_ylabel('Price')

ax2.bar(s.index, s['volume'], color='steelblue', alpha=0.7)
ax2.set_ylabel('Volume')

up_patch   = mpatches.Patch(color='green', label='Up day')
down_patch = mpatches.Patch(color='red',   label='Down day')
ax1.legend(handles=[up_patch, down_patch], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Technical indicators for one symbol
# Note: indicators need ≥ 20 bars to be meaningful; add more data if results look flat.
s = df[df.symbol == SYMBOL].copy().set_index('date').sort_index()
close = s['close']

s['sma20'] = SMAIndicator(close, window=20).sma_indicator()
s['sma50'] = SMAIndicator(close, window=min(50, len(s))).sma_indicator()
s['rsi14'] = RSIIndicator(close, window=14).rsi()

bb = BollingerBands(close, window=20)
s['bb_upper'] = bb.bollinger_hband()
s['bb_lower'] = bb.bollinger_lband()

macd_ind = MACD(close)
s['macd']        = macd_ind.macd()
s['macd_signal'] = macd_ind.macd_signal()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1, 1]})

# Price + Bollinger Bands + SMAs
axes[0].plot(s.index, s['close'],    label='Close',  linewidth=1.5)
axes[0].plot(s.index, s['sma20'],    label='SMA 20', linestyle='--', linewidth=1)
axes[0].plot(s.index, s['sma50'],    label='SMA 50', linestyle='--', linewidth=1)
axes[0].fill_between(s.index, s['bb_lower'], s['bb_upper'], alpha=0.1, label='BB band')
axes[0].set_title(f'{SYMBOL} — technical indicators')
axes[0].set_ylabel('Price')
axes[0].legend(fontsize=8)

# RSI
axes[1].plot(s.index, s['rsi14'], color='purple', linewidth=1)
axes[1].axhline(70, color='red',   linestyle=':', linewidth=0.8)
axes[1].axhline(30, color='green', linestyle=':', linewidth=0.8)
axes[1].set_ylabel('RSI 14')
axes[1].set_ylim(0, 100)

# MACD
axes[2].plot(s.index, s['macd'],        label='MACD',   linewidth=1)
axes[2].plot(s.index, s['macd_signal'], label='Signal', linewidth=1, linestyle='--')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Summary stats across all symbols
summary = (
    df.groupby('symbol')
    .agg(
        days     = ('date', 'count'),
        avg_close= ('close', 'mean'),
        avg_vol  = ('volume', 'mean'),
        high_52w = ('high', 'max'),
        low_52w  = ('low', 'min'),
    )
    .round(2)
    .sort_values('avg_vol', ascending=False)
)
summary